# 31 — Fan numbers

The tiers from notebook 30, turned into numbers a fan can picture: missed ball/strike calls per 1,000 calls, per game and per season. All the logic lives in `src/fan_metrics.py`.

Each season is measured against its own strike zone (old zone 2021–2025, new ABS zone 2026). A full season behind the plate is taken as 29 games, the median for a full-time umpire.

In [ ]:
import sys

import pandas as pd
from IPython.display import display

sys.path.insert(0, "../src")
import fan_metrics
import plot_style
import umpire_tiers

pd.set_option("display.max_columns", 40)

tables = fan_metrics.build()
umpire_tiers.save(tables)

by_tier = tables["fan_metrics_by_tier"]
gaps = tables["tier_gap_intervals"]
by_umpire = tables["fan_metrics_by_umpire"]
league = tables["league_missed_calls_by_season"]
next_season = tables["tier_next_season"]
SEASON = int(by_tier["season"].max())

## 1. The league: missed calls per season

Every graded ball/strike call, all umpires included.

In [ ]:
display(
    league[
        [
            "season",
            "zone",
            "umpires",
            "games",
            "called_pitches",
            "missed",
            "accuracy",
            "misses_per_game",
            "misses_per_1000",
        ]
    ].round(4)
)
fig = fan_metrics.plot_missed_calls(league)
path = plot_style.save_figure(fig, "missed_calls_by_season")

## 2. The tiers in fan numbers

`extra_..._vs_tier_1` is how many more calls a tier misses than Tier 1.

In [ ]:
columns = [
    "tier",
    "tier_name",
    "umpires",
    "accuracy",
    "misses_per_1000",
    "misses_per_game",
    "misses_per_season",
    "extra_misses_per_game_vs_tier_1",
    "extra_misses_per_season_vs_tier_1",
]
display(by_tier.loc[by_tier["season"] == SEASON, columns].round(2))
fig = fan_metrics.plot_misses_by_tier(by_tier, SEASON)
path = plot_style.save_figure(fig, "misses_per_game_by_tier")

## 3. Tier 1 against Tier 5, every season

The `low` and `high` columns are a 95% interval from resampling whole games.

In [ ]:
gaps.round(2)

## 4. What a tier is worth the next season

The honest test of a tier: how the same umpires did on new games the following year. `aal_this_season_pp` is their accuracy above league in the season they earned the tier; `aal_next_season_pp` is the next season.

In [ ]:
next_season.round(3)

## 5. Every umpire against a league-average umpire

`misses_vs_league` is how many more calls an umpire missed than a league-average umpire would have missed on the same number of calls (negative = fewer). `calls_to_reach_league` is what an umpire below the average would have needed.

In [ ]:
season = by_umpire[by_umpire["season"] == SEASON].sort_values("misses_vs_league")
show = [
    "official_name",
    "tier",
    "plate_games",
    "missed",
    "accuracy",
    "misses_per_game",
    "misses_vs_league",
    "calls_to_reach_league",
]
print("Fewest misses against a league-average umpire,", SEASON)
display(season[show].head(10).round(3))
print("Most misses against a league-average umpire,", SEASON)
display(season[show].tail(10).round(3))
below = season[season["calls_to_reach_league"] > 0]
bottom = season[season["tier"] >= 4]
print(len(below), "of", len(season), "umpires were below the league average")
print(
    "bottom quartile:",
    len(bottom),
    "umpires, needing",
    bottom["calls_to_reach_league"].min(),
    "to",
    bottom["calls_to_reach_league"].max(),
    "more correct calls each (average",
    round(bottom["calls_to_reach_league"].mean(), 1),
    ")",
)

## Takeaways

1. **The league misses fewer calls.** 28,988 missed calls in 2021 (old zone), 24,243 in 2026 (new zone): 4,745 fewer. Per game that is 11.9 down to 10.0, about two fewer missed calls a game.
2. **The 2026 tiers.** A Tier 1 umpire misses 8.2 calls a game (54 per 1,000 calls, about 239 a season). Tier 3 misses 10.0 (65; 289). Tier 5 misses 11.8 (76; 341).
3. **Tier 1 against Tier 5 in 2026:** 2.2 pp (95% interval 1.9 to 2.6), which is 22 fewer misses per 1,000 calls, 3.5 fewer a game (2.9 to 4.2) and about 103 a season (85 to 121).
4. **The gap has narrowed.** In 2021 it was 3.1 pp, 4.9 calls a game and about 142 a season.
5. **A tier carries over.** Tier 1 umpires were 1.3 pp above the league in the season they earned the tier and 0.7 pp above it the next season; 88% stayed above the league and 64% stayed in the top quartile. Tier 5 umpires were 1.5 pp below, then 0.8 pp below; 6% got above the league and 75% were in the bottom quartile again.
6. **Reaching the league average.** 42 of the 89 umpires were below the 2026 average. The 22 in the bottom quartile each needed 23 to 59 more correct calls over the season (38 on average), about one to two a game. At the other end, Ripperger and Visconti each missed about 64 fewer calls than a league-average umpire would have on the same calls.

Outputs: `fan_metrics_by_tier.csv`, `tier_gap_intervals.csv`, `fan_metrics_by_umpire.csv`, `league_missed_calls_by_season.csv`, `tier_next_season.csv` in `reports/tables`; `missed_calls_by_season.png`, `misses_per_game_by_tier.png` in `reports/figures`.